In [ ]:
"""
LLM Security Benchmark
This script measures how well an LLM classifies vulnerable and patched code pairs from CSV input.
"""

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from sklearn.metrics import classification_report, roc_curve, auc
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, auc
import numpy as np
import json
import os
import sys
import re
import time
import pandas as pd
from datetime import datetime
from pathlib import Path
from llmScan import *

In [ ]:
# Shared helpers for paired and unpaired benchmark CSVs
def _normalize_benchmark_label(value):
    if pd.isna(value):
        return "unknown"
    text = str(value).strip().lower()
    label_map = {
        "vulnerable": "vulnerable",
        "vuln": "vulnerable",
        "unsafe": "vulnerable",
        "malicious": "vulnerable",
        "safe": "safe",
        "patched": "safe",
        "benign": "safe",
        "non-vulnerable": "safe",
        "nonvulnerable": "safe",
        "not vulnerable": "safe",
        "true": "vulnerable",
        "false": "safe",
        "1": "vulnerable",
        "0": "safe",
    }
    return label_map.get(text, text or "unknown")


def _report_label(label):
    return "patched" if label == "safe" else label


def _detect_benchmark_input_schema(
    df,
    before_col="vulnerable_code",
    after_col="fixed_code",
    code_col=None,
    label_col=None,
):
    columns = set(df.columns)
    if before_col in columns and after_col in columns:
        return {"mode": "paired", "before_col": before_col, "after_col": after_col}

    code_candidates = [candidate for candidate in [code_col, "code_sample", "code", "sample_code", "source_code", "snippet", "content"] if candidate]
    label_candidates = [candidate for candidate in [label_col, "sample_label", "label", "ground_truth", "target", "class", "vulnerability_label", "status"] if candidate]

    resolved_code = next((candidate for candidate in code_candidates if candidate in columns), None)
    resolved_label = next((candidate for candidate in label_candidates if candidate in columns), None)

    if resolved_code and resolved_label:
        return {"mode": "single", "code_col": resolved_code, "label_col": resolved_label}

    raise ValueError(
        "Could not detect benchmark schema. Expected either paired columns "
        f"({before_col!r}, {after_col!r}) or a single code/label pair such as "
        "('code_sample', 'sample_label')."
    )


def _benchmark_result_mode(df, prefix):
    if f"{prefix}_classification_before" in df.columns and f"{prefix}_classification_after" in df.columns:
        return "paired"
    if f"{prefix}_classification" in df.columns and (
        f"{prefix}_expected_label" in df.columns or "expected_label" in df.columns
    ):
        return "single"
    return None


def _build_benchmark_report_data(df, prefix):
    mode = _benchmark_result_mode(df, prefix)
    if mode == "paired":
        total_rows = len(df)
        y_true = ["vulnerable"] * total_rows + ["patched"] * total_rows
        y_pred_raw = df[f"{prefix}_classification_before"].tolist() + df[f"{prefix}_classification_after"].tolist()
        y_pred = [_report_label(label) for label in y_pred_raw]
        return {"mode": mode, "y_true": y_true, "y_pred": y_pred, "total_predictions": total_rows * 2}
    if mode == "single":
        expected_col = f"{prefix}_expected_label" if f"{prefix}_expected_label" in df.columns else "expected_label"
        y_true = df[expected_col].tolist()
        y_pred_raw = df[f"{prefix}_classification"].tolist()
        y_true_report = [_report_label(label) for label in y_true]
        y_pred_report = [_report_label(label) for label in y_pred_raw]
        return {"mode": mode, "y_true": y_true_report, "y_pred": y_pred_report, "total_predictions": len(df)}
    return None

In [ ]:
# Create the benchmark-specific template and function
def create_benchmark_chain():
    """Create a chain specifically for benchmark boolean responses using Ollama structured output."""
    model, _ = get_model_and_chain()
    if model is None:
        return None, None

    # Reuse model metadata from llmScan
    benchmark_model_name = getattr(model, "model", PRIMARY_MODEL)
    benchmark_base_url = getattr(model, "base_url", None)

    benchmark_template = """<system>You are a security vulnerability analyzer for Python code.</system>

<user>
Your objective is to determine whether the provided Python code snippet is vulnerable or safe.
The snippet may be vulnerable or it may be safe. Classify the full snippet as a whole unit.

Examples of security vulnerabilities include:
- Unsafe use of untrusted input (e.g., from `request.args`, `request.form`, environment variables, external files)
- Dangerous function calls (e.g., `eval`, `exec`, `os.system`, `subprocess.run`, `pickle.loads`, `yaml.load`)
- Insecure file handling (e.g., `open` or `send_file` with user-controlled paths)
- Cryptographic mistakes (e.g., hardcoded keys, insecure algorithms)
- Web-specific issues (e.g., Cross-Site Scripting (XSS), CSRF vulnerabilities, Open Redirects)
- Hardcoded secrets (e.g., API keys, passwords, tokens)
- Misconfigurations (e.g., exposing debug mode, bad CORS policies)
- Omission of security features (e.g., missing input validation, hashing, secrets management)

Analyze this Python code snippet as a whole unit.
Do not score or classify line-by-line; return one final classification for the full snippet.
If any part of the snippet is meaningfully vulnerable, set `vulnerable` to true.

Review only the following code snippet:
```python
{python_code}
```
Ignore any context outside this code block.

Return only a structured JSON object with a single boolean field named `vulnerable`.
</user>
"""

    prompt = ChatPromptTemplate.from_template(benchmark_template)

    vulnerability_schema = {
        "type": "object",
        "properties": {
            "vulnerable": {"type": "boolean"}
        },
        "required": ["vulnerable"],
        "additionalProperties": False
    }

    llm_kwargs = {
        "model": benchmark_model_name,
    }
    if benchmark_base_url:
        llm_kwargs["base_url"] = benchmark_base_url

    structured_model = ChatOllama(**llm_kwargs).with_structured_output(vulnerability_schema)
    chain = prompt | structured_model

    return chain, benchmark_model_name

benchmark_chain, model_name = create_benchmark_chain()
print(f"Benchmark chain created successfully with model: {model_name}" if benchmark_chain else "Failed to create benchmark chain")

In [ ]:
# Define the benchmark test function with timing for code strings
def test_vulnerabilities_benchmark(python_code_text, include_raw_response=False, include_prompt=False):
    """
    Analyze a Python code string for security vulnerabilities with a boolean-style LLM response.

    Returns:
        tuple: (success, result, model_name, scan_time)
        tuple: (success, result, model_name, scan_time, raw_response) when include_raw_response=True
        tuple: (success, result, model_name, scan_time, raw_response, prompt) when include_prompt=True
    """
    start_time = time.time()

    def _format_return(success, result, model, scan_time, raw_response=None, prompt=None):
        if include_prompt:
            return success, result, model, scan_time, raw_response, prompt
        if include_raw_response:
            return success, result, model, scan_time, raw_response
        return success, result, model, scan_time

    try:
        if benchmark_chain is None:
            scan_time = time.time() - start_time
            return _format_return(False, None, "Error: No working LLM model available", scan_time, None, None)

        # Format the prompt to capture what the LLM receives
        formatted_prompt = None
        if include_prompt:
            try:
                # Get the prompt template from the chain and format it
                prompt_obj = benchmark_chain.first
                if hasattr(prompt_obj, 'format'):
                    formatted_prompt = prompt_obj.format(python_code=str(python_code_text or ""))
            except Exception as e:
                print(f"Could not format prompt for display: {e}")

        retries = 0
        response = None
        while retries < MAX_RETRIES:
            try:
                response = benchmark_chain.invoke({"python_code": str(python_code_text or "")})
                break
            except Exception as e:
                retries += 1
                print(f"LLM invocation failed (attempt {retries}/{MAX_RETRIES}): {e}")
                if retries >= MAX_RETRIES:
                    scan_time = time.time() - start_time
                    return _format_return(False, None, model_name, scan_time, None, formatted_prompt)
                print(f"Retrying in {RETRY_DELAY} seconds...")
                time.sleep(RETRY_DELAY)

        scan_time = time.time() - start_time

        if isinstance(response, dict):
            if "vulnerable" in response and isinstance(response["vulnerable"], bool):
                return _format_return(True, response["vulnerable"], model_name, scan_time, response, formatted_prompt)
            # Backward-compatible fallback for misspelled key if a model emits it.
            if "vulenrable" in response and isinstance(response["vulenrable"], bool):
                return _format_return(True, response["vulenrable"], model_name, scan_time, response, formatted_prompt)
            print(f"Structured response missing boolean 'vulnerable': {response}")
            return _format_return(False, None, model_name, scan_time, response, formatted_prompt)

        if isinstance(response, bool):
            return _format_return(True, response, model_name, scan_time, response, formatted_prompt)
        if isinstance(response, str):
            response_lower = response.strip().lower()
            if response_lower == "true":
                return _format_return(True, True, model_name, scan_time, response, formatted_prompt)
            if response_lower == "false":
                return _format_return(True, False, model_name, scan_time, response, formatted_prompt)
            if "true" in response_lower and "false" not in response_lower:
                return _format_return(True, True, model_name, scan_time, response, formatted_prompt)
            if "false" in response_lower and "true" not in response_lower:
                return _format_return(True, False, model_name, scan_time, response, formatted_prompt)
            print(f"Ambiguous response: {response}")
            return _format_return(False, None, model_name, scan_time, response, formatted_prompt)

        print(f"Unexpected response type: {type(response)}")
        return _format_return(False, None, model_name, scan_time, response, formatted_prompt)

    except Exception:
        scan_time = time.time() - start_time
        return _format_return(False, None, model_name, scan_time, None, None)


def label_from_prediction(success, predicted_bool):
    if not success or predicted_bool is None:
        return "unknown"
    return "vulnerable" if predicted_bool else "safe"


print("test_vulnerabilities_benchmark function defined successfully!")


In [ ]:
# Define CSV-driven benchmark runner
def run_benchmark_from_csv(
    csv_path,
    before_col="vulnerable_code",
    after_col="fixed_code",
    code_col=None,
    label_col=None,
    verbose=True,
):
    """
    Run the LLM benchmark from a CSV in either paired or single-sample form.

    Paired input: one row contains vulnerable and patched snippets.
    Single-sample input: one row contains one code sample plus its label.
    """
    df = pd.read_csv(csv_path)

    schema = _detect_benchmark_input_schema(
        df,
        before_col=before_col,
        after_col=after_col,
        code_col=code_col,
        label_col=label_col,
    )

    output_df = df.copy()
    output_df["benchmark_input_mode"] = schema["mode"]
    model_names = []

    if schema["mode"] == "paired":
        before_predictions = []
        after_predictions = []
        before_correct = []
        after_correct = []
        before_success = []
        after_success = []
        before_completed = []
        after_completed = []
        before_times = []
        after_times = []

        for idx, row in output_df.iterrows():
            before_code = row.get(schema["before_col"], "")
            after_code = row.get(schema["after_col"], "")

            b_completed, b_pred, b_model, b_time = test_vulnerabilities_benchmark(before_code)
            a_completed, a_pred, a_model, a_time = test_vulnerabilities_benchmark(after_code)

            b_label = label_from_prediction(b_completed, b_pred)
            a_label = label_from_prediction(a_completed, a_pred)

            expected_before = "vulnerable"
            expected_after = "safe"
            b_is_correct = b_label == expected_before
            a_is_correct = a_label == expected_after

            before_predictions.append(b_label)
            after_predictions.append(a_label)
            before_correct.append(b_is_correct)
            after_correct.append(a_is_correct)
            before_success.append(b_is_correct)
            after_success.append(a_is_correct)
            before_completed.append(b_completed)
            after_completed.append(a_completed)
            before_times.append(round(b_time, 3))
            after_times.append(round(a_time, 3))
            model_names.append(a_model or b_model or "unknown")

            if verbose:
                print(
                    f"Row {idx + 1}: before={b_label} (expected vulnerable), "
                    f"after={a_label} (expected safe)"
                )

        output_df["expected_before"] = "vulnerable"
        output_df["expected_after"] = "safe"
        output_df["llm_classification_before"] = before_predictions
        output_df["llm_classification_after"] = after_predictions
        output_df["llm_correct_before"] = before_correct
        output_df["llm_correct_after"] = after_correct
        output_df["llm_success_before"] = before_success
        output_df["llm_success_after"] = after_success
        output_df["llm_completed_before"] = before_completed
        output_df["llm_completed_after"] = after_completed
        output_df["llm_scan_time_before_seconds"] = before_times
        output_df["llm_scan_time_after_seconds"] = after_times
        output_df["llm_pair_correct"] = output_df["llm_correct_before"] & output_df["llm_correct_after"]
    else:
        code_column = schema["code_col"]
        label_column = schema["label_col"]
        predictions = []
        correct = []
        success = []
        completed = []
        scan_times = []
        expected_labels = []

        for idx, row in output_df.iterrows():
            code_text = row.get(code_column, "")
            expected_label = _normalize_benchmark_label(row.get(label_column, ""))

            row_completed, row_pred, row_model, row_time = test_vulnerabilities_benchmark(code_text)
            row_label = label_from_prediction(row_completed, row_pred)

            is_correct = row_label == expected_label

            expected_labels.append(expected_label)
            predictions.append(row_label)
            correct.append(is_correct)
            success.append(is_correct)
            completed.append(row_completed)
            scan_times.append(round(row_time, 3))
            model_names.append(row_model or "unknown")

            if verbose:
                print(
                    f"Row {idx + 1}: expected={expected_label} | "
                    f"predicted={row_label} | completed={row_completed}"
                )

        output_df["expected_label"] = expected_labels
        output_df["llm_expected_label"] = expected_labels
        output_df["llm_classification"] = predictions
        output_df["llm_correct"] = correct
        output_df["llm_success"] = success
        output_df["llm_completed"] = completed
        output_df["llm_scan_time_seconds"] = scan_times
        output_df["llm_code_column"] = code_column
        output_df["llm_label_column"] = label_column

    output_df["llm_model"] = model_names

    return output_df


print("run_benchmark_from_csv function defined successfully!")

In [ ]:
# Run the CSV benchmark
input_csv_path = "splitData.csv"

print(f"Starting LLM CSV Security Benchmark on {input_csv_path} with model {model_name}...")

df_results = run_benchmark_from_csv(input_csv_path, verbose=False)

date_stamp = datetime.now().strftime("%Y%m%d")
raw_model = str(model_name) if model_name else "LLM"
model_tag = re.sub(r"[^A-Za-z0-9]+", "", raw_model) or "LLM"
output_file = f"benchmark{model_tag}{date_stamp}.csv"
df_results.to_csv(output_file, index=False)

print(f"Benchmark completed for {len(df_results)} rows.")
print(f"Results saved to: {output_file}")

In [ ]:
# Calculate and display comprehensive metrics
print("=" * 60)
print("LLM CSV BENCHMARK RESULTS")
print("=" * 60)

if any(c.startswith("llm_") for c in df_results.columns):
    metric_prefix = "llm"
elif any(c.startswith("bandit_") for c in df_results.columns):
    metric_prefix = "bandit"
else:
    metric_prefix = None

total_rows = len(df_results)
print(f"Total rows analyzed: {total_rows}")

if metric_prefix is None:
    print("No supported metric columns found (expected llm_* or bandit_*).")
else:
    report_data = _build_benchmark_report_data(df_results, metric_prefix)

    if report_data is None:
        print("No supported classification columns found.")
    else:
        mode = report_data["mode"]
        y_true = report_data["y_true"]
        y_pred = report_data["y_pred"]
        total_predictions = report_data["total_predictions"]

        if mode == "paired":
            failed_before = int((df_results[f"{metric_prefix}_completed_before"] != True).sum()) if f"{metric_prefix}_completed_before" in df_results.columns and total_rows else 0
            failed_after = int((df_results[f"{metric_prefix}_completed_after"] != True).sum()) if f"{metric_prefix}_completed_after" in df_results.columns and total_rows else 0
            tests_failed = failed_before + failed_after
            print(f"Unsuccessful tests: {tests_failed}/{total_predictions}")
        else:
            failed_single = int((df_results[f"{metric_prefix}_completed"] != True).sum()) if f"{metric_prefix}_completed" in df_results.columns and total_rows else 0
            print(f"Unsuccessful tests: {failed_single}/{total_predictions}")

        print("\n" + "-" * 60)
        print("Classification Report (total predictions)")
        print("-" * 60)
        print(classification_report(y_true, y_pred, labels=["vulnerable", "patched"], target_names=["vulnerable", "patched"], zero_division=0, digits=3))

        # Build confusion matrix directly from y_true/y_pred to avoid correctness-column artifacts.
        pred_series = pd.Series(y_pred, name="Predicted")
        pred_series = pred_series.where(pred_series.isin(["vulnerable", "patched"]), "unknown")
        actual_series = pd.Series(y_true, name="Actual")

        confusion_matrix = pd.crosstab(actual_series, pred_series, dropna=False)
        display_cols = ["vulnerable", "patched", "unknown"]
        present_cols = [c for c in display_cols if c in confusion_matrix.columns]
        confusion_matrix = confusion_matrix.reindex(index=["vulnerable", "patched"], columns=present_cols, fill_value=0)
        confusion_matrix.index = [f"Actual: {idx}" for idx in confusion_matrix.index]
        confusion_matrix.columns = [f"Predicted: {col}" for col in confusion_matrix.columns]

        print("-" * 60)
        print("Confusion Matrix (total predictions)")
        print("-" * 60)
        print(confusion_matrix.to_string())
        print("-" * 60)
        print("Legend: rows=actual labels, columns=predicted labels")
        print("=" * 60)

        # ROC curve from hard-label predictions (unknown treated as non-vulnerable score 0).
        y_true_binary = [1 if label == "vulnerable" else 0 for label in y_true]
        y_score = [1 if label == "vulnerable" else 0 for label in y_pred]
        if len(set(y_true_binary)) > 1:
            fpr, tpr, _ = roc_curve(y_true_binary, y_score)
            roc_auc = auc(fpr, tpr)
            print(f"ROC AUC: {roc_auc:.3f}")
            plt.figure(figsize=(6, 5))
            plt.plot(fpr, tpr, linewidth=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
            plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline")
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title("ROC Curve")
            plt.legend(loc="lower right")
            plt.grid(alpha=0.3)
            plt.show()
        else:
            print("ROC AUC: unavailable (only one class in ground truth).")

        if mode == "paired":
            before_completed = df_results[f"{metric_prefix}_completed_before"].mean() if f"{metric_prefix}_completed_before" in df_results.columns and total_rows else 0
            after_completed = df_results[f"{metric_prefix}_completed_after"].mean() if f"{metric_prefix}_completed_after" in df_results.columns and total_rows else 0
            print(f"Completion rate (before-code): {before_completed:.2%}")
            print(f"Completion rate (after-code): {after_completed:.2%}")

            avg_before_time = df_results[f"{metric_prefix}_scan_time_before_seconds"].mean() if f"{metric_prefix}_scan_time_before_seconds" in df_results.columns and total_rows else 0
            avg_after_time = df_results[f"{metric_prefix}_scan_time_after_seconds"].mean() if f"{metric_prefix}_scan_time_after_seconds" in df_results.columns and total_rows else 0
            total_before_time = df_results[f"{metric_prefix}_scan_time_before_seconds"].sum() if f"{metric_prefix}_scan_time_before_seconds" in df_results.columns and total_rows else 0
            total_after_time = df_results[f"{metric_prefix}_scan_time_after_seconds"].sum() if f"{metric_prefix}_scan_time_after_seconds" in df_results.columns and total_rows else 0
            total_time = total_before_time + total_after_time

            print(f"\nTiming Statistics:")
            print(f"Before-code - Total: {total_before_time:.3f}s | Average: {avg_before_time:.3f}s")
            print(f"After-code  - Total: {total_after_time:.3f}s | Average: {avg_after_time:.3f}s")
            print(f"Total scan time: {total_time:.3f}s")
        else:
            completion_rate = df_results[f"{metric_prefix}_completed"].mean() if f"{metric_prefix}_completed" in df_results.columns and total_rows else 0
            avg_time = df_results[f"{metric_prefix}_scan_time_seconds"].mean() if f"{metric_prefix}_scan_time_seconds" in df_results.columns and total_rows else 0
            total_time = df_results[f"{metric_prefix}_scan_time_seconds"].sum() if f"{metric_prefix}_scan_time_seconds" in df_results.columns and total_rows else 0

            print(f"Completion rate: {completion_rate:.2%}")
            print(f"\nTiming Statistics:")
            print(f"Total: {total_time:.3f}s | Average: {avg_time:.3f}s")

        if total_rows:
            model_col = f"{metric_prefix}_model"
            if model_col in df_results.columns:
                models_used = sorted(df_results[model_col].dropna().unique().tolist())
                if models_used:
                    print(f"\nModel(s) in results: {', '.join(models_used)}")

In [ ]:
# Load and analyze benchmark results from a saved CSV file
from sklearn.metrics import classification_report, roc_curve, auc
import matplotlib.pyplot as plt

def analyze_benchmark_csv(csv_path):
    """
    Load a previously generated benchmark CSV and calculate all metrics.

    Supports both paired benchmark outputs and single-sample benchmark outputs.
    """
    try:
        df = pd.read_csv(csv_path)

        print("=" * 60)
        print(f"ANALYZING BENCHMARK CSV: {csv_path}")
        print("=" * 60)

        total_rows = len(df)
        print(f"Total rows/pairs analyzed: {total_rows}")

        if any(c.startswith("llm_") for c in df.columns):
            metric_prefix = "llm"
        elif any(c.startswith("bandit_") for c in df.columns):
            metric_prefix = "bandit"
        else:
            metric_prefix = None

        if metric_prefix is None:
            print("No supported metric columns found (expected llm_* or bandit_*).")
            return

        report_data = _build_benchmark_report_data(df, metric_prefix)
        if report_data is None:
            print("No supported classification columns found.")
            return

        y_true = report_data["y_true"]
        y_pred = report_data["y_pred"]
        total_predictions = report_data["total_predictions"]

        if report_data["mode"] == "paired":
            completed_before_col = f"{metric_prefix}_completed_before"
            completed_after_col = f"{metric_prefix}_completed_after"
            if completed_before_col in df.columns and completed_after_col in df.columns:
                failed_before = int((df[completed_before_col] != True).sum()) if total_rows else 0
                failed_after = int((df[completed_after_col] != True).sum()) if total_rows else 0
                tests_failed = failed_before + failed_after
                print(f"Unsuccessful tests: {tests_failed}/{total_predictions}")
        else:
            completed_col = f"{metric_prefix}_completed"
            if completed_col in df.columns:
                tests_failed = int((df[completed_col] != True).sum()) if total_rows else 0
                print(f"Unsuccessful tests: {tests_failed}/{total_predictions}")

        print("\n" + "-" * 60)
        print("Classification Report (total predictions)")
        print("-" * 60)
        print(classification_report(y_true, y_pred, labels=["vulnerable", "patched"], target_names=["vulnerable", "patched"], zero_division=0, digits=3))

        pred_series = pd.Series(y_pred, name="Predicted")
        pred_series = pred_series.where(pred_series.isin(["vulnerable", "patched"]), "unknown")
        actual_series = pd.Series(y_true, name="Actual")

        confusion_matrix = pd.crosstab(actual_series, pred_series, dropna=False)
        display_cols = ["vulnerable", "patched", "unknown"]
        present_cols = [c for c in display_cols if c in confusion_matrix.columns]
        confusion_matrix = confusion_matrix.reindex(index=["vulnerable", "patched"], columns=present_cols, fill_value=0)
        confusion_matrix.index = [f"Actual: {idx}" for idx in confusion_matrix.index]
        confusion_matrix.columns = [f"Predicted: {col}" for col in confusion_matrix.columns]

        print("-" * 60)
        print("Confusion Matrix (total predictions)")
        print("-" * 60)
        print(confusion_matrix.to_string())
        print("-" * 60)
        print("Legend: rows=actual labels, columns=predicted labels")
        print("=" * 60)

        y_true_binary = [1 if label == "vulnerable" else 0 for label in y_true]
        y_score = [1 if label == "vulnerable" else 0 for label in y_pred]
        if len(set(y_true_binary)) > 1:
            fpr, tpr, _ = roc_curve(y_true_binary, y_score)
            roc_auc = auc(fpr, tpr)
            print(f"ROC AUC: {roc_auc:.3f}")
            plt.figure(figsize=(6, 5))
            plt.plot(fpr, tpr, linewidth=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
            plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline")
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title("ROC Curve")
            plt.legend(loc="lower right")
            plt.grid(alpha=0.3)
            plt.show()
        else:
            print("ROC AUC: unavailable (only one class in ground truth).")

        if report_data["mode"] == "paired":
            before_completed_col = f"{metric_prefix}_completed_before"
            after_completed_col = f"{metric_prefix}_completed_after"
            before_completed = df[before_completed_col].mean() if before_completed_col in df.columns and total_rows else 0
            after_completed = df[after_completed_col].mean() if after_completed_col in df.columns and total_rows else 0
            print(f"Completion rate (before-code): {before_completed:.2%}")
            print(f"Completion rate (after-code): {after_completed:.2%}")

            before_time_col = f"{metric_prefix}_scan_time_before_seconds"
            after_time_col = f"{metric_prefix}_scan_time_after_seconds"
            avg_before_time = df[before_time_col].mean() if before_time_col in df.columns and total_rows else 0
            avg_after_time = df[after_time_col].mean() if after_time_col in df.columns and total_rows else 0
            total_before_time = df[before_time_col].sum() if before_time_col in df.columns and total_rows else 0
            total_after_time = df[after_time_col].sum() if after_time_col in df.columns and total_rows else 0
            total_time = total_before_time + total_after_time
            print(f"\nTiming Statistics:")
            print(f"Before-code - Total: {total_before_time:.3f}s | Average: {avg_before_time:.3f}s")
            print(f"After-code  - Total: {total_after_time:.3f}s | Average: {avg_after_time:.3f}s")
            print(f"Total scan time: {total_time:.3f}s")
        else:
            completed_col = f"{metric_prefix}_completed"
            time_col = f"{metric_prefix}_scan_time_seconds"
            if completed_col in df.columns:
                completion_rate = df[completed_col].mean() if total_rows else 0
                print(f"Completion rate: {completion_rate:.2%}")
            if time_col in df.columns:
                avg_time = df[time_col].mean() if total_rows else 0
                total_time = df[time_col].sum() if total_rows else 0
                print(f"\nTiming Statistics:")
                print(f"Total: {total_time:.3f}s | Average: {avg_time:.3f}s")

        model_col = f"{metric_prefix}_model"
        if model_col in df.columns:
            models_used = sorted(df[model_col].dropna().unique().tolist())
            if models_used:
                print(f"\nModel(s) in results: {', '.join(models_used)}")

        print("=" * 60)

    except FileNotFoundError:
        print(f"Error: Could not find file at {csv_path}")
    except Exception as e:
        print(f"Error reading CSV: {e}")

input_csv_path = "splitData.csv"

# Check if file exists before running
import os
if os.path.exists(input_csv_path):
    analyze_benchmark_csv(input_csv_path)
else:
    print(f"File not found: {input_csv_path}")
    print("Available CSV files in results/:")
    if os.path.exists("results"):
        for f in os.listdir("results"):
            if f.endswith(".csv"):
                print(f"  - {f}")

In [ ]:
# Compare benchmark stats across multiple CSV files with a single combined ROC plot
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, auc
import numpy as np
import re

def compare_benchmark_csvs(csv_paths):
    """
    Compare benchmark performance across multiple CSV files and plot one combined ROC chart.

    Supports both paired benchmark outputs and single-sample benchmark outputs.
    """
    def _clean_label_from_filename(path_str):
        stem = Path(path_str).stem
        stem = re.sub(r"^benchmark", "", stem, flags=re.IGNORECASE)
        stem = re.sub(r"\d{8}$", "", stem)
        stem = stem.strip("_- ")
        return stem or Path(path_str).stem

    def _parse_model_label(df, metric_prefix, path_str):
        candidate_cols = [
            f"{metric_prefix}_model",
            "tool_name",
            "model_name",
            "model",
            "analyzer",
            "engine",
            "scanner",
        ]

        parsed_values = []
        for col in candidate_cols:
            if col in df.columns:
                values = sorted(df[col].dropna().astype(str).str.strip().unique().tolist())
                values = [v for v in values if v]
                if values:
                    parsed_values.extend(values)
                    break

        if parsed_values:
            return ", ".join(parsed_values)

        stem_lower = Path(path_str).stem.lower()
        has_bandit_cols = any(c.startswith("bandit_") for c in df.columns)
        if metric_prefix == "bandit" or has_bandit_cols or "bandit" in stem_lower:
            return "Bandit"

        return _clean_label_from_filename(path_str)

    if not csv_paths:
        print("No CSV files provided.")
        return pd.DataFrame()

    plt.figure(figsize=(8, 6))
    roc_drawn = False
    summaries = []

    for csv_path in csv_paths:
        try:
            df = pd.read_csv(csv_path)
        except FileNotFoundError:
            print(f"File not found: {csv_path}")
            continue
        except Exception as e:
            print(f"Error reading {csv_path}: {e}")
            continue

        total_rows = len(df)
        if total_rows == 0:
            print(f"Skipping empty file: {csv_path}")
            continue

        if any(c.startswith("llm_") for c in df.columns):
            metric_prefix = "llm"
        elif any(c.startswith("bandit_") for c in df.columns):
            metric_prefix = "bandit"
        else:
            print(f"Skipping {csv_path}: no llm_* or bandit_* columns found")
            continue

        report_data = _build_benchmark_report_data(df, metric_prefix)
        if report_data is None:
            print(f"Skipping {csv_path}: missing benchmark classification columns")
            continue

        y_true = report_data["y_true"]
        y_pred = report_data["y_pred"]
        total_predictions = report_data["total_predictions"]
        y_true_binary = np.array([1 if label == "vulnerable" else 0 for label in y_true])
        y_score = np.array([1 if label == "vulnerable" else 0 for label in y_pred])

        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=["vulnerable"],
            average=None,
            zero_division=0,
        )
        precision_v = float(precision[0]) if len(precision) else 0.0
        recall_v = float(recall[0]) if len(recall) else 0.0
        f1_v = float(f1[0]) if len(f1) else 0.0

        if report_data["mode"] == "paired":
            completed_before_col = f"{metric_prefix}_completed_before"
            completed_after_col = f"{metric_prefix}_completed_after"
            if completed_before_col in df.columns and completed_after_col in df.columns:
                failed_before = int((df[completed_before_col] != True).sum())
                failed_after = int((df[completed_after_col] != True).sum())
                tests_failed = failed_before + failed_after
            else:
                tests_failed = int(sum(1 for p in y_pred if p == "unknown"))
        else:
            completed_col = f"{metric_prefix}_completed"
            if completed_col in df.columns:
                tests_failed = int((df[completed_col] != True).sum())
            else:
                tests_failed = int(sum(1 for p in y_pred if p == "unknown"))

        model_label = _parse_model_label(df, metric_prefix, csv_path)

        roc_auc = None
        if len(np.unique(y_true_binary)) > 1:
            fpr, tpr, _ = roc_curve(y_true_binary, y_score)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, linewidth=2, label=f"{model_label} (AUC={roc_auc:.3f})")
            roc_drawn = True

        summaries.append({
            "file": csv_path,
            "model_label": model_label,
            "rows": total_rows,
            "predictions": total_predictions,
            "accuracy": round(float(accuracy), 4),
            "precision_vulnerable": round(precision_v, 4),
            "recall_vulnerable": round(recall_v, 4),
            "f1_vulnerable": round(f1_v, 4),
            "tests_failed": tests_failed,
            "roc_auc": round(float(roc_auc), 4) if roc_auc is not None else None,
        })

    if not summaries:
        print("No valid benchmark CSV files were processed.")
        return pd.DataFrame()

    summary_df = pd.DataFrame(summaries).sort_values(by=["roc_auc", "accuracy"], ascending=False, na_position="last")
    print("=" * 80)
    print("MULTI-CSV BENCHMARK COMPARISON")
    print("=" * 80)
    print(summary_df.to_string(index=False))
    print("=" * 80)

    if roc_drawn:
        plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Combined ROC Curve Across Benchmark CSVs")
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        plt.show()
    else:
        print("ROC plot unavailable: insufficient class variation in processed files.")

    return summary_df

# Example usage: compare multiple benchmark outputs in one table + one ROC chart
csv_files_to_compare = [
    "results/benchmarkBandit20260405.csv",
    "results/benchmarkphi414b20260416.csv",
    "results/benchmarkllama318b20260408.csv",
    "results/benchmarkmistralnemo12b20260410.csv",
    "results/benchmarkgemma312b20260412.csv",
]

# Keep this call enabled to run immediately, or comment it out if you only want the function.
comparison_df = compare_benchmark_csvs(csv_files_to_compare)

In [ ]:
# Optional compact summary export
if len(df_results):
    if any(c.startswith("llm_") for c in df_results.columns):
        metric_prefix = "llm"
    else:
        metric_prefix = "bandit"
    report_data = _build_benchmark_report_data(df_results, metric_prefix)
    if report_data is None:
        summary = {
            "total_row_pairs": len(df_results),
            "benchmark_mode": "unknown",
        }
    elif report_data["mode"] == "paired":
        summary = {
            "total_row_pairs": len(df_results),
            "benchmark_mode": "paired",
            "before_accuracy": float(df_results[f"{metric_prefix}_correct_before"].mean()) if f"{metric_prefix}_correct_before" in df_results.columns else 0.0,
            "after_accuracy": float(df_results[f"{metric_prefix}_correct_after"].mean()) if f"{metric_prefix}_correct_after" in df_results.columns else 0.0,
            "pair_accuracy": float(df_results[f"{metric_prefix}_pair_correct"].mean()) if f"{metric_prefix}_pair_correct" in df_results.columns else 0.0,
            "before_success_rate": float(df_results[f"{metric_prefix}_success_before"].mean()) if f"{metric_prefix}_success_before" in df_results.columns else 0.0,
            "after_success_rate": float(df_results[f"{metric_prefix}_success_after"].mean()) if f"{metric_prefix}_success_after" in df_results.columns else 0.0,
            "before_completed_rate": float(df_results[f"{metric_prefix}_completed_before"].mean()) if f"{metric_prefix}_completed_before" in df_results.columns else 0.0,
            "after_completed_rate": float(df_results[f"{metric_prefix}_completed_after"].mean()) if f"{metric_prefix}_completed_after" in df_results.columns else 0.0,
            "tests_failed": int((df_results[f"{metric_prefix}_completed_before"] != True).sum() + (df_results[f"{metric_prefix}_completed_after"] != True).sum()) if f"{metric_prefix}_completed_before" in df_results.columns and f"{metric_prefix}_completed_after" in df_results.columns else 0,
            "unknown_before": int((df_results[f"{metric_prefix}_classification_before"] == "unknown").sum()) if f"{metric_prefix}_classification_before" in df_results.columns else 0,
            "unknown_after": int((df_results[f"{metric_prefix}_classification_after"] == "unknown").sum()) if f"{metric_prefix}_classification_after" in df_results.columns else 0,
        }
    else:
        summary = {
            "total_row_pairs": len(df_results),
            "benchmark_mode": "single",
            "accuracy": float(df_results[f"{metric_prefix}_correct"].mean()) if f"{metric_prefix}_correct" in df_results.columns else 0.0,
            "success_rate": float(df_results[f"{metric_prefix}_success"].mean()) if f"{metric_prefix}_success" in df_results.columns else 0.0,
            "completed_rate": float(df_results[f"{metric_prefix}_completed"].mean()) if f"{metric_prefix}_completed" in df_results.columns else 0.0,
            "tests_failed": int((df_results[f"{metric_prefix}_completed"] != True).sum()) if f"{metric_prefix}_completed" in df_results.columns else 0,
            "unknown_predictions": int((df_results[f"{metric_prefix}_classification"] == "unknown").sum()) if f"{metric_prefix}_classification" in df_results.columns else 0,
        }
else:
    summary = {
        "total_row_pairs": 0,
        "benchmark_mode": "empty",
        "accuracy": 0.0,
        "success_rate": 0.0,
        "completed_rate": 0.0,
        "tests_failed": 0,
        "unknown_predictions": 0,
    }

summary_df = pd.DataFrame([summary])
summary_file = "llm_csv_benchmark_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"Summary metrics saved to: {summary_file}")

In [ ]:
# Debug LLM Input/Output from CSV rows
def debug_llm_from_csv(
    csv_path,
    sample_pairs=1,
    start_row=0,
    random_sample=False,
    random_seed=42,
    before_col="vulnerable_code",
    after_col="fixed_code",
    code_col=None,
    label_col=None,
    preview_chars=240,
):
    df = pd.read_csv(csv_path)
    if df.empty:
        print("CSV is empty.")
        return

    schema = _detect_benchmark_input_schema(
        df,
        before_col=before_col,
        after_col=after_col,
        code_col=code_col,
        label_col=label_col,
    )

    total_rows = len(df)
    sample_pairs = int(sample_pairs) if sample_pairs is not None else 1
    if sample_pairs <= 0:
        print("sample_pairs must be >= 1")
        return

    if random_sample:
        sample_n = min(sample_pairs, total_rows)
        sampled_df = df.sample(n=sample_n, random_state=random_seed)
        selected_indices = sampled_df.index.tolist()
    else:
        if start_row < 0 or start_row >= total_rows:
            print(f"start_row {start_row} is out of range (0-{total_rows-1})")
            return
        end_row = min(start_row + sample_pairs, total_rows)
        selected_indices = list(range(start_row, end_row))

    print("=" * 60)
    if schema["mode"] == "paired":
        print(f"LLM CSV DEBUG | Testing {len(selected_indices)} pair(s)")
        print(f"Columns: before={schema['before_col']}, after={schema['after_col']}")
    else:
        print(f"LLM CSV DEBUG | Testing {len(selected_indices)} sample(s)")
        print(f"Columns: code={schema['code_col']}, label={schema['label_col']}")
    print("=" * 60)

    for i, row_index in enumerate(selected_indices, start=1):
        row = df.iloc[row_index]
        print("-" * 60)
        print(f"Sample {i}/{len(selected_indices)} | Row {row_index + 1}")

        if schema["mode"] == "paired":
            before_code = str(row.get(schema["before_col"], "") or "")
            after_code = str(row.get(schema["after_col"], "") or "")
            print(f"Source lengths: before={len(before_code)} chars, after={len(after_code)} chars")
            print(f"Line counts: before={before_code.count(chr(10)) + 1 if before_code else 0}, after={after_code.count(chr(10)) + 1 if after_code else 0}")
            print(f"Before/after identical: {before_code == after_code}")
            print(f"Before preview:\n{before_code[:preview_chars]}")
            if len(before_code) > preview_chars:
                print("... [before preview truncated]")
            print(f"After preview:\n{after_code[:preview_chars]}")
            if len(after_code) > preview_chars:
                print("... [after preview truncated]")

            b_completed, b_pred, b_model, b_time, b_raw, b_prompt = test_vulnerabilities_benchmark(before_code, include_raw_response=True, include_prompt=True)
            a_completed, a_pred, a_model, a_time, a_raw, a_prompt = test_vulnerabilities_benchmark(after_code, include_raw_response=True, include_prompt=True)
            b_label = label_from_prediction(b_completed, b_pred)
            a_label = label_from_prediction(a_completed, a_pred)
            b_success = b_label == "vulnerable"
            a_success = a_label == "safe"

            print("\n" + "=" * 60)
            print("FULL PROMPT RECEIVED BY LLM (BEFORE CODE):")
            print("=" * 60)
            print(b_prompt if b_prompt else "[Prompt unavailable]")
            print("=" * 60)
            print(f"Before expected: vulnerable | predicted: {b_label} | success: {b_success} | completed: {b_completed} | model: {b_model} | time: {b_time:.3f}s")
            print(f"Before raw response: {repr(b_raw)}")

            print("\n" + "=" * 60)
            print("FULL PROMPT RECEIVED BY LLM (AFTER CODE):")
            print("=" * 60)
            print(a_prompt if a_prompt else "[Prompt unavailable]")
            print("=" * 60)
            print(f"After expected: safe       | predicted: {a_label} | success: {a_success} | completed: {a_completed} | model: {a_model} | time: {a_time:.3f}s")
            print(f"After raw response: {repr(a_raw)}")
        else:
            code_text = str(row.get(schema["code_col"], "") or "")
            expected_label = _normalize_benchmark_label(row.get(schema["label_col"], ""))
            print(f"Source length: code={len(code_text)} chars")
            print(f"Line count: {code_text.count(chr(10)) + 1 if code_text else 0}")
            print(f"Code preview:\n{code_text[:preview_chars]}")
            if len(code_text) > preview_chars:
                print("... [code preview truncated]")

            completed, pred, model, scan_time, raw_response, prompt = test_vulnerabilities_benchmark(
                code_text,
                include_raw_response=True,
                include_prompt=True,
            )
            predicted_label = label_from_prediction(completed, pred)
            success = predicted_label == expected_label

            print("\n" + "=" * 60)
            print("FULL PROMPT RECEIVED BY LLM:")
            print("=" * 60)
            print(prompt if prompt else "[Prompt unavailable]")
            print("=" * 60)
            print(
                f"Expected: {expected_label} | predicted: {predicted_label} | success: {success} | "
                f"completed: {completed} | model: {model} | time: {scan_time:.3f}s"
            )
            print(f"Raw response: {repr(raw_response)}")

# Example debug usage
input_csv_path = "splitData.csv"
# debug_llm_from_csv(input_csv_path, sample_pairs=3, start_row=0)
debug_llm_from_csv(input_csv_path, sample_pairs=5, random_sample=True, random_seed=42, preview_chars=240)